In [11]:
import torch
from oracle.presets import get_model
from torchinfo import summary

In [12]:
model = get_model("BTSv2")

In [21]:
# ...existing code...
batch_size = 16
# Ensure feature dim (last dim) matches your model (BTSv2 GRU expects 5)
feat_dim = 5

batch = {
    "ts": torch.randn(batch_size, 100, feat_dim),
    "length": torch.full((batch_size,), 10, dtype=torch.int64),
    "static": torch.randn(batch_size, 30),
}

class ONNXWrapper(torch.nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base

    def forward(self, x_ts, lengths, x_static):
        # Keys must match model.forward() expectation ("ts","length","static")
        batch = {"ts": x_ts, "length": lengths, "static": x_static}
        return self.base(batch)

model.eval()
wrapper = ONNXWrapper(model)

# 1. Generate Summary
summary(
    wrapper,
    input_data=(batch["ts"], batch["length"], batch["static"]),
    col_names=["input_size", "output_size", "num_params"],
    depth=4,
)

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
ONNXWrapper                              [16, 100, 5]              [16, 10]                  --
├─GRU_MD_Improved: 1-1                   [16, 100, 5]              [16, 10]                  --
│    └─GRU: 2-1                          [160, 5]                  [160, 256]                400,128
│    └─Linear: 2-2                       [16, 10, 256]             [16, 10, 128]             32,896
│    └─Linear: 2-3                       [16, 10, 128]             [16, 10, 1]               128
│    └─Linear: 2-4                       [16, 256]                 [16, 128]                 32,896
│    └─LayerNorm: 2-5                    [16, 128]                 [16, 128]                 256
│    └─GELU: 2-6                         [16, 128]                 [16, 128]                 --
│    └─Dropout: 2-7                      [16, 128]                 [16, 128]                 --
│    └─Linear: 2-8  

In [ ]:
# import torch
# from oracle.presets import get_model

# model = get_model("BTSv2")

# batch_size = 16
# batch = {
#     "x_ts": torch.randn(batch_size, 100, 5),
#     "lengths": torch.full((batch_size,), 10, dtype=torch.int64),
#     "x_static": torch.randn(batch_size, 30),
# }

# class ONNXWrapper(torch.nn.Module):
#     def __init__(self, base):
#         super().__init__()
#         self.base = base

#     def forward(self, x_ts, lengths, x_static):
#         batch = {"ts": x_ts, "length": lengths, "static": x_static}
#         return self.base(batch)

# wrapper = ONNXWrapper(model.eval())
# torch.onnx.export(
#     wrapper,
#     (batch["x_ts"], batch["lengths"], batch["x_static"]),
#     "model.onnx",
#     input_names=["x_ts", "lengths", "x_static"],
#     output_names=["logits"],
#     opset_version=11,
#     operator_export_type=torch.onnx.OperatorExportTypes.ONNX_ATEN_FALLBACK,
# )

/Users/vedshah/anaconda3/envs/VT/lib/python3.11/site-packages/torch/onnx/utils.py:478: FutureWarning: Setting `operator_export_type` to something other than default is deprecated. The option will be removed in a future release.
  warnings.warn(
[W204 18:14:16.760378000 shape_type_inference.cpp:1999] Warning: The shape inference of prim::PackPadded type is missing, so it may result in wrong shape inference for the exported graph. Please consider adding it in symbolic function. (function UpdateReliable)
[W204 18:14:16.762480000 shape_type_inference.cpp:1999] Warning: The shape inference of prim::PackPadded type is missing, so it may result in wrong shape inference for the exported graph. Please consider adding it in symbolic function. (function UpdateReliable)
/Users/vedshah/anaconda3/envs/VT/lib/python3.11/site-packages/torch/onnx/symbolic_opset9.py:4277: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with GRU can cause an error when runnin